In [ ]:
"""
Walmart-Amazon Sample and Ground Truth Generation Script
Shuffles the original JSON dataset, saves it as a CSV, and then generates
adjacent positive pairs within clusters to create the ground truth dataset.
"""

import json
import os
from pathlib import Path
import pandas as pd

project_root = Path("").resolve()
os.chdir(project_root)
# ============================================================
# 1. Shuffle and Create Sample Dataset
# ============================================================
print("--- Creating Shuffled Sample ---")
# Load the original JSON file
original_json_path = './dataset/walmart_amazon_unique.json'
df = pd.read_json(original_json_path)

# Shuffle the rows randomly (frac=1 shuffles 100% of the rows)
# random_state ensures reproducibility, remove it for completely random results
df_shuffled = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Ensure output directory exists
Path('data').mkdir(parents=True, exist_ok=True)

# Save the shuffled data to a CSV file
sample_csv_path = './dataset/sample_walmart_amazon.csv'
df_shuffled.to_csv(sample_csv_path, index=False)
print(f"Shuffled data saved as '{sample_csv_path}'.\n")


# ============================================================
# 2. Ground Truth Generation
# ============================================================
# Paths Configuration
INPUT_DATA_PATH = Path(sample_csv_path) 
OUTPUT_GT_PATH = "./dataset/sample_walmart_amazon_gt.csv"

def load_data(file_path: Path) -> pd.DataFrame:
    """Helper function to load data based on file extension."""
    if not file_path.exists():
        raise FileNotFoundError(f"Could not find the input file at: {file_path}")

    if file_path.suffix.lower() == ".json":
        print(f"Loading JSON data from {file_path}...")
        with open(file_path, "r", encoding="utf-8") as f:
            return pd.DataFrame(json.load(f))
    elif file_path.suffix.lower() in [".csv", ".tsv"]:
        sep = "\t" if file_path.suffix.lower() == ".tsv" else ","
        print(f"Loading delimited data from {file_path}...")
        return pd.read_csv(file_path, sep=sep)
    else:
        raise ValueError(f"Unsupported file format: {file_path.suffix}")


def generate_ground_truth_adjacent_pairs(input_path: Path, output_csv_path: Path):
    """Generates a ground truth dataframe containing linear adjacent positive pairs.

    Columns produced: ['ltable_id', 'rtable_id']
    """
    df = load_data(input_path)

    # Validate required keys match your dataset structure
    required_cols = {"id", "cluster_id"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns in {input_path.name}: {missing}")

    gt_rows = []

    # Reset index to implicitly capture and preserve original row/file order
    df = df.reset_index(drop=False).rename(columns={"index": "_row_order"})

    # Group by cluster, keeping the original sequence order intact
    for cluster_id, group in df.groupby("cluster_id", sort=False):
        group = group.sort_values("_row_order")
        ids = group["id"].tolist()

        # If a cluster has n items, generate exactly (n - 1) adjacent sequential links
        if len(ids) >= 2:
            for i in range(len(ids) - 1):
                gt_rows.append(
                    {"ltable_id": ids[i], "rtable_id": ids[i + 1]}
                )

    # Build final Ground Truth Dataframe
    gt_df = pd.DataFrame(gt_rows, columns=["ltable_id", "rtable_id"])

    # Save to CSV format
    gt_df.to_csv(output_csv_path, index=False)

    print(f"\n--- Ground Truth Generation Complete ---")
    print(f"Source items processed from: {input_path}")
    print(f"Saved GT mapping file to: {output_csv_path}")
    print(f"Total positive matching pairs generated: {len(gt_df)}")

    return gt_df


if __name__ == "__main__":
    # Generate Ground Truth mapping
    gt_dataframe = generate_ground_truth_adjacent_pairs(
        INPUT_DATA_PATH, OUTPUT_GT_PATH
    )

    # Preview the output
    print("\nGenerated Ground Truth Preview:")
    print(gt_dataframe.head(10))